# 🔧 Module 9: Feature Engineering
## Theory + Practical

---

## 📚 THEORY

> **"Garbage in, garbage out"** — Even the best model fails with bad features.

**Feature Engineering** = Creating, transforming, and selecting the right input features.

### Key Techniques

| Technique | What it does | Example |
|-----------|-------------|--------|
| **Scaling** | Normalize feature range | StandardScaler, MinMaxScaler |
| **Encoding** | Convert categories to numbers | One-Hot, Label, Ordinal |
| **Imputation** | Fill missing values | Mean, Median, Mode |
| **Feature Selection** | Remove irrelevant features | SelectKBest, Importance |

### Scaling: When to Use Which?
```
StandardScaler  → Mean=0, Std=1  → Use for most algorithms
MinMaxScaler    → Range [0,1]    → Use for neural networks, KNN
RobustScaler    → Uses median    → Use when data has outliers
```

### Encoding Categories
```
Color = [Red, Blue, Green]

Label Encoding:  Red=0, Blue=1, Green=2  ← Bad! Implies order
One-Hot:         Red=[1,0,0], Blue=[0,1,0], Green=[0,0,1]  ← Correct
```

---

## 🔬 PRACTICAL

In [ ]:
# Install all required packages
!pip install numpy pandas matplotlib seaborn scikit-learn scipy -q
print('Packages ready ✅')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.datasets import load_breast_cancer
import warnings; warnings.filterwarnings('ignore')

print('Libraries loaded ✅')

### Practical 1: Handling Missing Values

In [ ]:
# Simulate a messy dataset
np.random.seed(42)
df = pd.DataFrame({
    'age':    [25, np.nan, 35, 42, np.nan, 30, 28, np.nan, 55, 33],
    'salary': [50000, 60000, np.nan, 80000, 45000, np.nan, 55000, 70000, 90000, 65000],
    'score':  [7, 8, 6, np.nan, 9, 7, np.nan, 8, 6, 9]
})

print('Original data (with missing values):')
print(df)
print(f'\nMissing values:\n{df.isnull().sum()}')

In [ ]:
# Different imputation strategies
for strategy in ['mean', 'median', 'most_frequent']:
    imputer = SimpleImputer(strategy=strategy)
    df_filled = pd.DataFrame(imputer.fit_transform(df), columns=df.columns)
    print(f'\nAfter {strategy} imputation:')
    print(df_filled.round(1))

### Practical 2: Encoding Categorical Variables

In [ ]:
df_cat = pd.DataFrame({
    'color': ['red', 'blue', 'green', 'red', 'blue'],
    'size':  ['small', 'large', 'medium', 'large', 'small'],
    'price': [10, 20, 15, 12, 18]
})

print('Original:')
print(df_cat)

# One-Hot Encoding (nominal: no order)
df_ohe = pd.get_dummies(df_cat, columns=['color'], prefix='color')
print('\nAfter One-Hot Encoding (color):')
print(df_ohe)

# Ordinal Encoding (ordinal: has order)
size_order = {'small': 0, 'medium': 1, 'large': 2}
df_cat['size_encoded'] = df_cat['size'].map(size_order)
print('\nAfter Ordinal Encoding (size):')
print(df_cat[['size', 'size_encoded']])

### Practical 3: Feature Selection

In [ ]:
cancer = load_breast_cancer()
X, y = cancer.data, cancer.target
print(f'Original features: {X.shape[1]}')

# Method 1: SelectKBest (statistical test)
selector = SelectKBest(score_func=f_classif, k=10)
X_kbest = selector.fit_transform(X, y)
selected_names = np.array(cancer.feature_names)[selector.get_support()]
print('\nTop 10 features (SelectKBest):')
for f in selected_names:
    print(f'  - {f}')

# Method 2: Random Forest Feature Importance
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X, y)
importance = pd.Series(rf.feature_importances_, index=cancer.feature_names)
top5 = importance.nlargest(5)

plt.figure(figsize=(8, 3))
top5.sort_values().plot(kind='barh', color='steelblue', edgecolor='black')
plt.title('Top 5 Features by Random Forest Importance')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

### Practical 4: Scaling Comparison

In [ ]:
data = np.array([[1, 200], [2, 250], [5, 100], [100, 300], [3, 220]], dtype=float)

scalers = {
    'Original': None,
    'StandardScaler': StandardScaler(),
    'MinMaxScaler': MinMaxScaler(),
    'RobustScaler': RobustScaler()
}

for name, scaler in scalers.items():
    result = data if scaler is None else scaler.fit_transform(data)
    print(f'{name}:')
    print(f'  A → min={result[:,0].min():.2f}, max={result[:,0].max():.2f}, mean={result[:,0].mean():.2f}')
    print(f'  B → min={result[:,1].min():.2f}, max={result[:,1].max():.2f}, mean={result[:,1].mean():.2f}')
    print()

### 🏋️ Try It Yourself!

In [ ]:
# ✏️ YOUR TURN: Change n_features and see accuracy change
n_features = 15  # Try: 5, 10, 15, 20, 30

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('selector', SelectKBest(f_classif, k=n_features)),
    ('model', LogisticRegression(max_iter=1000))
])

scores = cross_val_score(pipe, X, y, cv=5, scoring='accuracy')
print(f'Features selected: {n_features}')
print(f'CV Accuracy: {scores.mean():.4f} ± {scores.std():.4f}')

## 🎓 Summary

| Technique | When |
|-----------|------|
| StandardScaler | Most models (SVM, LR, KNN) |
| MinMaxScaler | Neural networks, image data |
| RobustScaler | Data with outliers |
| One-Hot Encoding | Nominal categories (no order) |
| Ordinal Encoding | Ordered categories (small<medium<large) |
| SelectKBest | Quick statistical feature selection |
| Feature Importance | Model-based feature selection |

➡️ **Next: Ensemble Methods (Boosting)**